In [ ]:
import sem
import matplotlib.pyplot as plt
import seaborn as sns
import pprint
sns.set_style("whitegrid")
import random
import pandas as pd
import io
from io import StringIO

In [ ]:
# This cell may take few times to build if first usage (also, change_parallel_process to 1 if you are not using gpu)
import sem
ns_path = 'ns-3-dev/'
script = 'ai-ns3'
campaign_dir = 'testResults-ai'
campaign = sem.CampaignManager.new(
    ns_path=ns_path,
    script=script,
    campaign_dir=campaign_dir,
    overwrite=True,
    max_parallel_processes=8
)


In [ ]:
print(campaign)

In [ ]:
#check the input of the datarates and delays
params = {
    'LinkDataRate' :["100Kbps","500Kbps","1Mbps","5Mbps","10Mbps","15Mbps"], 
    'LinkDelay'    :["2ms","10ms","50ms","100ms","200ms","500ms","1000ms"],
    'AppPacketSize' :["1000","1024","1400","1500"],
    'MaxPackets' : ["10","20","30","40","50"],
    'Interval' : [0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]


}
runs = 2 #specify how many randomized experiments we want sem to perform for each parameter combination

campaign.run_missing_simulations(params, runs=runs) #this will run 24 experiments

In [ ]:
campaign.db.get_complete_results()

In [ ]:
print("There are %s results in the database\n" % len(list(campaign.db.get_complete_results())))
example_result = campaign.db.get_complete_results()[0]
print("This is an example result:\n")
pprint.pprint(example_result)

In [ ]:
# Extract CSV string from output
csv_str = example_result['output']['network_data.csv']

# Load CSV string into a DataFrame
df = pd.read_csv(StringIO(csv_str), index_col=False)

# Add experiment metadata
df['experiment_id'] = example_result['meta']['id']
df['exitcode'] = example_result['meta']['exitcode'] #use to verify that the file was correctly executed (return 0)
df['elapsed_time'] = example_result['meta']['elapsed_time']

#  Add parameters columns (flatten the params dict into DataFrame columns)
for key, value in example_result['params'].items():
    df[key] = value

# Now df contains all CSV data rows plus experiment ID and parameter columns
df.head()

In [ ]:
import pandas as pd

big_df_list = []  # list to collect smaller dataframes

for example_result in campaign.db.get_complete_results():
    # Extract CSV string from output
    csv_str = example_result['output']['network_data.csv']

    #  Load CSV string into a DataFrame
    df = pd.read_csv(StringIO(csv_str), index_col=False)

    # Add experiment metadata
    df['experiment_id'] = example_result['meta']['id']
    df['exitcode'] = example_result['meta']['exitcode'] #use to verify that the file was correctly executed (return 0)
    df['elapsed_time'] = example_result['meta']['elapsed_time']

    # Step 4: Add parameters columns (flatten the params dict into DataFrame columns)
    for key, value in example_result['params'].items():
        df[key] = value

    
    big_df_list.append(df)

# After the loop, concatenate all at once
big_df = pd.concat(big_df_list, ignore_index=True)
#big_df.shape #the number of rows =sum (number of flows per experiment) e.g. is there are 2 flows for 16800 experiments, the df will have 33600 rows

In [ ]:
big_df.head()

In [ ]:
big_df.columns

In [ ]:
big_df[['MeanDelay(s)', 'MeanJitter(s)', 'MeanRxBitrate(bps)']].describe()

In [ ]:
sns.histplot(data=big_df, x='MeanJitter(s)', bins=30, kde=False)
plt.xlabel('MeanJitter(s)')
plt.ylabel('Count')
plt.title('Histogram of MeanJitter(s)')
plt.show()

In [ ]:
sns.catplot(data=big_df,
            x='LinkDelay',
            y='MeanRxBitrate(bps)',
            kind='point',
            order = params['LinkDelay'],#order=["100Kbps","500Kbps","1Mbps","5Mbps","10Mbps","15Mbps"],
            height=3,
            aspect=2)  # Width = height * aspect, so 5*2 = 10)
plt.show()

In [ ]:
#decomment for visualization 
# y variables (metrics you want to plot)

cvars = ['MeanDelay(s)' ]  #  'MeanTxBitrate(bps)'

# Number of subplots = number of x_vars * number of cvars
n_rows = len(params)
n_cols = len(cvars)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), squeeze=False)

for i, x_var in enumerate(params.keys()):
    order = params[x_var]
    for j, y_var in enumerate(cvars):
        ax = axes[i][j]

        sns.pointplot(
            data=big_df,
            x=x_var,
            y=y_var,
            order=order,
            ax=ax,
            dodge=True,
            markers='o',
            linestyles='-'
        )

        ax.set_title(f"{y_var} vs {x_var}")
        ax.set_xlabel(x_var)
        ax.set_ylabel(y_var)

        # Optional: rotate x labels if needed
        ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()



In [ ]:
import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier